# P2: Station Geography + Soil Merge

Primary merge per `merge.md` (§2, **P2**). Assembles the static, per-station
context table: every EPA monitoring station with its watershed (HUC-12/10/8),
its SSURGO soil map unit, and that map unit's soil attributes. Grain is one row
per `MonitoringLocationIdentifier` — nothing here varies with time.

**Inputs:**
- `data/tabular/02_clean/water-quality/epa-stations-clean.csv` — station identity/geography (base table)
- `data/spatial/02_clean/nhdplus/wbd-huc12-station-crosswalk-clean.csv` — station → HUC-12/10/8, joined on `MonitoringLocationIdentifier`
- `data/spatial/02_clean/ssurgo/ssurgo-mapunit-station-crosswalk-clean.csv` — station → `mukey`, joined on `MonitoringLocationIdentifier`
- `data/tabular/02_clean/soil/ssurgo-iowa-attributes-clean.csv` — soil attributes, joined on `mukey`

A derived 5-digit `county_fips` (`StateCode` + `CountyCode`, zero-padded) is added
so downstream secondary merges can attach county-grain agriculture data.

**Output:** `data/03a_merge_primary/station-geo-soil-clean.csv`, one row per station.

> Soil coverage is ~65% by design: stations sit in/next to streams and lakes, so
> the rest land in non-soil water map units (`map_unit_symbol` `W`/`RIVER`/`LAKE`)
> that carry no soil component — an expected property of station placement, not a
> join defect (see `DATA.md`, ssurgo crosswalk coverage note).

In [1]:
import os

import pandas as pd

CLEAN = "../../data/tabular/02_clean"
SPATIAL = "../../data/spatial/02_clean"
OUT_DIR = "../../data/03a_merge_primary"
OUT_FILE = f"{OUT_DIR}/station-geo-soil.csv"

KEY = "MonitoringLocationIdentifier"

## Step 1: Load stations + derive `county_fips`

The stations table is the base grain. `county_fips` isn't present verbatim — it's
the 2-digit `StateCode` (`19` = Iowa) concatenated with the zero-padded 3-digit
`CountyCode`, giving the standard 5-digit FIPS the agriculture tables key on.

In [2]:
df = pd.read_csv(f"{CLEAN}/water-quality/epa-stations-clean.csv")
print(f"Stations: {df.shape}")
assert not df.duplicated(subset=[KEY]).any(), "Station grain violated: duplicate station rows"

# Derived 5-digit county FIPS = StateCode (2) + CountyCode (3), zero-padded.
has_fips = df["StateCode"].notna() & df["CountyCode"].notna()
df["county_fips"] = pd.NA
df.loc[has_fips, "county_fips"] = (
    df.loc[has_fips, "StateCode"].astype(int).astype(str).str.zfill(2)
    + df.loc[has_fips, "CountyCode"].astype(int).astype(str).str.zfill(3)
)
print(f"county_fips derived for {has_fips.sum():,} / {len(df):,} stations")
print(f"Distinct counties: {df['county_fips'].nunique()}")

Stations: (1666, 10)
county_fips derived for 1,666 / 1,666 stations
Distinct counties: 99


## Step 2: Watershed crosswalk (HUC-12/10/8)

Left join the spatial WBD crosswalk on `MonitoringLocationIdentifier`, attaching
the HUC-12/10/8 the station falls in. The crosswalk's `huc8_code` matches the
station table's own `HUCEightDigitCode` (verified 0 mismatches during cleaning),
but the crosswalk additionally carries the finer HUC-12/10 needed downstream.

In [3]:
df_huc = pd.read_csv(
    f"{SPATIAL}/nhdplus/wbd-huc12-station-crosswalk-clean.csv",
    dtype={"huc12_code": str, "huc10_code": str, "huc8_code": str},
)
assert not df_huc.duplicated(subset=[KEY]).any(), "HUC crosswalk is not 1 row/station"

df = df.merge(df_huc, on=KEY, how="left")
print(f"Merged shape: {df.shape}")
print(f"HUC-12 match rate: {df['huc12_code'].notna().mean():.1%}")
assert len(df) == df[KEY].nunique(), "HUC join fanned out station rows"

Merged shape: (1666, 16)
HUC-12 match rate: 100.0%


## Step 3: Soil map unit + attributes

Two joins: first the spatial map-unit crosswalk (station → `mukey`) on
`MonitoringLocationIdentifier`, then the soil attributes on `mukey`. The
crosswalk already carries the station-polygon's `map_unit_symbol` and
`survey_area` (covering 100% of stations, including the `W`/`RIVER`/`LAKE`
water units), so those two columns are dropped from the attributes table before
joining to avoid a redundant collision — the descriptive/quantitative soil
columns (`map_unit_name`, `hydrologic_group`, `ksat_mean`, …) come only from the
attributes side and are `NaN` for the ~35% of stations in water map units.

In [4]:
df_soil_x = pd.read_csv(
    f"{SPATIAL}/ssurgo/ssurgo-mapunit-station-crosswalk-clean.csv"
).rename(columns={"match_method": "soil_match_method", "match_distance_m": "soil_match_distance_m"})
assert not df_soil_x.duplicated(subset=[KEY]).any(), "Soil crosswalk is not 1 row/station"

df = df.merge(df_soil_x, on=KEY, how="left")
assert len(df) == df[KEY].nunique(), "Soil crosswalk join fanned out station rows"

# Attributes: drop the two keys already supplied by the crosswalk to avoid collision.
df_attrs = pd.read_csv(f"{CLEAN}/soil/ssurgo-iowa-attributes-clean.csv").drop(
    columns=["survey_area", "map_unit_symbol"]
)
assert not df_attrs.duplicated(subset=["mukey"]).any(), "Soil attributes are not 1 row/mukey"

df = df.merge(df_attrs, on="mukey", how="left")
print(f"Merged shape: {df.shape}")
print(f"mukey assigned:        {df['mukey'].notna().mean():.1%} of stations")
print(f"Soil attributes match: {df['hydrologic_group'].notna().mean():.1%} of stations")
assert len(df) == df[KEY].nunique(), "Soil attributes join fanned out station rows"

Merged shape: (1666, 28)
mukey assigned:        100.0% of stations
Soil attributes match: 64.6% of stations


## Step 4: Final checks and save

In [5]:
print(f"Final shape: {df.shape}")
print(f"Final columns: {list(df.columns)}")
assert not df.duplicated(subset=[KEY]).any(), "Output grain violated: duplicate station rows"

print("\nCoverage:")
print(f"  HUC-12 watershed:  {df['huc12_code'].notna().mean():.1%}")
print(f"  Soil map unit:     {df['mukey'].notna().mean():.1%}")
print(f"  Soil attributes:   {df['hydrologic_group'].notna().mean():.1%}")
print(f"  county_fips:       {df['county_fips'].notna().mean():.1%}")

df.head(3)

Final shape: (1666, 28)
Final columns: ['OrganizationIdentifier', 'MonitoringLocationIdentifier', 'MonitoringLocationName', 'MonitoringLocationTypeName', 'HUCEightDigitCode', 'LatitudeMeasure', 'LongitudeMeasure', 'StateCode', 'CountyCode', 'ProviderName', 'county_fips', 'huc12_code', 'huc12_name', 'huc10_code', 'huc8_code', 'huc12_acres', 'mukey', 'map_unit_symbol', 'survey_area', 'soil_match_method', 'soil_match_distance_m', 'map_unit_name', 'dominant_component', 'dominant_component_pct', 'hydrologic_group', 'drainage_class', 'ksat_mean', 'awc_mean']

Coverage:
  HUC-12 watershed:  100.0%
  Soil map unit:     100.0%
  Soil attributes:   64.6%
  county_fips:       100.0%


,OrganizationIdentifier,MonitoringLocationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,HUCEightDigitCode,LatitudeMeasure,LongitudeMeasure,StateCode,CountyCode,ProviderName,...,survey_area,soil_match_method,soil_match_distance_m,map_unit_name,dominant_component,dominant_component_pct,hydrologic_group,drainage_class,ksat_mean,awc_mean
0,USGS-IA,USGS-05387490,"Dry Run Creek near Decorah, IA",Stream,7060002,43.291361,-91.809321,19,191,NWIS,...,IA191,within,0.0,"Anthroportic Udorthents, 2 to 9 percent slopes",Anthroportic Udorthents,100.0,C,Moderately well drained,1.85,0.200
1,USGS-IA,USGS-05411260,"North Cedar Creek near Clayton, IA",Stream,7060003,42.962764,-91.229297,19,43,NWIS,...,IA043,within,0.0,"Dorchester-Volney complex, 1 to 5 percent slopes",Dorchester,50.0,B,Moderately well drained,9.00,0.217
2,USGS-IA,USGS-05412400,"Volga River at Littleport, IA",Stream,7060004,42.753875,-91.369025,19,43,NWIS,...,IA043,within,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 1,666 rows x 28 cols -> ../../data/03a_merge_primary/station-geo-soil.csv
